# PyStars Tutorial

PyStars automates significance testing for biological and life sciences data.
Given a pandas DataFrame with your measurements, it walks a decision flowchart
(normality → equal variance → test selection) so you don't have to.

```
%%uv
pystars
```

In [1]:
import numpy as np
import pandas as pd

import pystars as ps

---

## 1. Creating example data

Let's create three synthetic datasets we'll use throughout:

In [2]:
rng = np.random.default_rng(42)

# Two-group independent with a small effect
n = 30
df_2group = pd.DataFrame(
    {
        "group": ["control"] * n + ["treatment"] * n,
        "value": np.r_[rng.normal(10, 2, n), rng.normal(11.5, 2, n)],
    }
)

# Three-group with unequal variance (Welch ANOVA scenario)
df_3group = pd.DataFrame(
    {
        "group": ["A"] * 25 + ["B"] * 25 + ["C"] * 25,
        "value": np.r_[rng.normal(10, 1, 25), rng.normal(12, 2, 25), rng.normal(14, 3, 25)],
    }
)

# Paired / two-factor data
subjects = [f"mouse_{i}" for i in range(15)]
df_paired = pd.DataFrame(
    {
        "subject": subjects * 2,
        "group": ["pre"] * 15 + ["post"] * 15,
        "value": np.r_[rng.normal(10, 1.5, 15), rng.normal(8, 1.5, 15)],
    }
)

---

## 2. The auto-dispatcher: `pystars.test()`

The main entry point. Pass a DataFrame, tell it which column holds the
measurement (`value`) and which holds the group labels (`group`). PyStars
checks normality & equal variance and picks the right test automatically.

In [3]:
result = ps.test(df_2group, value="value", group="group")
result.show()

╭─────────────────────────────────────────────── Student's t-test ────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Student's t-test                                                                                    │
│  Statistic  -4.148                                                                                              │
│  p-value    0.0001111                                                                                           │
│  cohen_d    1.071                                                                                               │
│  CI95%      [-2.51, -0.88]                                                                                      │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.4161  not rejected                                                                 │
│  equal variance  levene    0.8273  not rejected                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
# Tidy one-row DataFrame export
result.to_dataframe()

,test,statistic,p_value,cohen_d,CI95%_0,CI95%_1,normality_method,normality_statistic,normality_p,equal_variance_method,equal_variance_statistic,equal_variance_p,posthoc
0,Student's t-test,-4.147649,0.000111,1.070918,-2.51,-0.88,shapiro,0.965142,0.416055,levene,0.048031,0.827294,<NA>


The dispatcher also works with more than two groups and with multi-factor designs.

In [5]:
# Three groups → Kruskal-Wallis or ANOVA depending on normality
result = ps.test(df_3group, value="value", group="group")
result.show()

╭───────────────────────────────────────────────── Welch's ANOVA ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Welch's ANOVA                                                                                       │
│  Statistic  31.29                                                                                               │
│  p-value    <0.0001                                                                                             │
│  np2        0.4345                                                                                              │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.1981  not rejected                                                                 │
│  equal variance  levene   <0.0001  rejected                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Paired test with the dispatcher

For paired designs, pass `subject` and `paired=True`:

In [6]:
result = ps.test(df_paired, value="value", group="group", subject="subject", paired=True)
result.show()

╭───────────────────────────────────────────────── Paired t-test ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Paired t-test                                                                                       │
│  Statistic  -3.363                                                                                              │
│  p-value    0.00464                                                                                             │
│  cohen_d    1.328                                                                                               │
│  CI95%      [-3.87, -0.86]                                                                                      │
│  Assumption  Method   p-value  Verdict                                                                          │
│  normality   shapiro    0.867  not rejected                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Plain-text summary (good for logging)
print(result.summary())

Test: Paired t-test
Statistic: -3.363
p-value: 0.00464
Effect size: cohen_d=1.328, CI95%=[-3.87, -0.86]
Assumptions:
  normality (shapiro): p=0.867 (not rejected)


### Controlling the significance level

The `alpha` parameter controls the threshold for assumption checks and
post-hoc gating (default 0.05):

In [8]:
ps.test(df_2group, value="value", group="group", alpha=0.01).show()

╭─────────────────────────────────────────────── Student's t-test ────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Student's t-test                                                                                    │
│  Statistic  -4.148                                                                                              │
│  p-value    0.0001111                                                                                           │
│  cohen_d    1.071                                                                                               │
│  CI95%      [-2.51, -0.88]                                                                                      │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.4161  not rejected                                                                 │
│  equal variance  levene    0.8273  not rejected                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Suppressing post-hoc tests

Set `auto_posthoc=False` to skip post-hoc tests after a significant ANOVA:

In [9]:
ps.test(df_3group, value="value", group="group", auto_posthoc=False).show()

╭───────────────────────────────────────────────── Welch's ANOVA ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Welch's ANOVA                                                                                       │
│  Statistic  31.29                                                                                               │
│  p-value    <0.0001                                                                                             │
│  np2        0.4345                                                                                              │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.1981  not rejected                                                                 │
│  equal variance  levene   <0.0001  rejected                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 3. Direct test functions

Each individual test is available as a standalone function. This bypasses the
flowchart and runs the test directly.

In [10]:
# Welch's t-test (default, safer for biological data)
r1 = ps.ttest(df_2group, value="value", group="group")

# Student's t-test (assumes equal variance)
r2 = ps.ttest(df_2group, value="value", group="group", welch=False)

# Paired t-test
r3 = ps.ttest(df_paired, value="value", group="group", subject="subject", paired=True)

# Mann-Whitney U (non-parametric, two independent groups)
r4 = ps.mannwhitney(df_2group, value="value", group="group")

# Wilcoxon signed-rank (non-parametric, paired)
r5 = ps.wilcoxon(df_paired, value="value", group="group", subject="subject")

# One-way ANOVA
r6 = ps.anova(df_3group, value="value", group="group")

# Welch's ANOVA (unequal variances)
r7 = ps.anova(df_3group, value="value", group="group", welch=True)

# Kruskal-Wallis (non-parametric, any number of groups)
r8 = ps.kruskal(df_3group, value="value", group="group")

In [11]:
# Compare them side by side
ps.to_dataframe([r1, r2, r3, r4, r5, r6, r7, r8])

,test,statistic,p_value,cohen_d,CI95%_0,CI95%_1,posthoc,CLES,RBC,np2,epsilon_squared
0,Welch's t-test,-4.147649,1.111906e-04,1.070918,-2.51,-0.88,<NA>,NaN,NaN,NaN,NaN
1,Student's t-test,-4.147649,1.110503e-04,1.070918,-2.51,-0.88,<NA>,NaN,NaN,NaN,NaN
2,Paired t-test,-3.363254,4.639983e-03,1.328331,-3.87,-0.86,<NA>,NaN,NaN,NaN,NaN
3,Mann-Whitney U test,210.000000,3.988102e-04,NaN,NaN,NaN,<NA>,0.233333,-0.533333,NaN,NaN
4,Wilcoxon signed-rank test,14.000000,6.713867e-03,NaN,NaN,NaN,<NA>,0.173333,-0.766667,NaN,NaN
5,One-way ANOVA,27.660031,1.223724e-09,NaN,NaN,NaN,<NA>,NaN,NaN,0.434496,NaN
6,Welch's ANOVA,31.292906,8.151327e-09,NaN,NaN,NaN,<NA>,NaN,NaN,0.434496,NaN
7,Kruskal-Wallis test,31.875200,1.197811e-07,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,0.430746


### Multiple-comparison correction

When you run many tests, correct the primary p-values across the batch. `holm` is a good conservative default for a small set of planned comparisons; use `fdr_bh` for larger exploratory screens.

In [12]:
# Correct p-values across this batch of direct tests
ps.to_dataframe([r1, r2, r3, r4, r5, r6, r7, r8], p_adjust="holm")

,test,statistic,p_value,p_adjusted,reject,p_adjust_method,p_adjust_alpha,cohen_d,CI95%_0,CI95%_1,posthoc,CLES,RBC,np2,epsilon_squared
0,Welch's t-test,-4.147649,1.111906e-04,5.552514e-04,True,holm,0.05,1.070918,-2.51,-0.88,<NA>,NaN,NaN,NaN,NaN
1,Student's t-test,-4.147649,1.110503e-04,5.552514e-04,True,holm,0.05,1.070918,-2.51,-0.88,<NA>,NaN,NaN,NaN,NaN
2,Paired t-test,-3.363254,4.639983e-03,9.279966e-03,True,holm,0.05,1.328331,-3.87,-0.86,<NA>,NaN,NaN,NaN,NaN
3,Mann-Whitney U test,210.000000,3.988102e-04,1.196431e-03,True,holm,0.05,NaN,NaN,NaN,<NA>,0.233333,-0.533333,NaN,NaN
4,Wilcoxon signed-rank test,14.000000,6.713867e-03,9.279966e-03,True,holm,0.05,NaN,NaN,NaN,<NA>,0.173333,-0.766667,NaN,NaN
5,One-way ANOVA,27.660031,1.223724e-09,9.789795e-09,True,holm,0.05,NaN,NaN,NaN,<NA>,NaN,NaN,0.434496,NaN
6,Welch's ANOVA,31.292906,8.151327e-09,5.705929e-08,True,holm,0.05,NaN,NaN,NaN,<NA>,NaN,NaN,0.434496,NaN
7,Kruskal-Wallis test,31.875200,1.197811e-07,7.186865e-07,True,holm,0.05,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,0.430746


---

## 4. Assumption checks

You can check normality and equal variance independently:

In [13]:
# Normality (Shapiro-Wilk) — per group
norm = ps.check_normality(df_2group, value="value", group="group")
norm.show()

# The per-group details are in .details
norm.details

╭────────────────────────────────────────── Shapiro-Wilk normality test ──────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Shapiro-Wilk normality test                                                                         │
│  Statistic  0.9651                                                                                              │
│  p-value    0.4161                                                                                              │
│                 Details                                                                                         │
│ ┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓                                                                        │
│ ┃ group     ┃ W      ┃ p      ┃ normal ┃                                                                        │
│ ┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩                                                                        │
│ │ control   │ 0.9651 │ 0.4161 │ True   │                                                                        │
│ │ treatment │ 0.9771 │ 0.7444 │ True   │                                                                        │
│ └───────────┴────────┴────────┴────────┘                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,group,W,p,normal
0,control,0.965142,0.416055,True
1,treatment,0.977101,0.744367,True


In [14]:
# Equal variance (Levene's test, median-centred)
var = ps.check_equal_variance(df_2group, value="value", group="group")
var.show()

╭─────────────────────────────────────── Levene's test for equal variance ────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Levene's test for equal variance                                                                    │
│  Statistic  0.04803                                                                                             │
│  p-value    0.8273                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
# Normality of paired differences
ps.check_normality(df_paired, value="value", group="group", subject="subject", paired=True).show()

╭────────────────────────────────────────── Shapiro-Wilk normality test ──────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Shapiro-Wilk normality test                                                                         │
│  Statistic  0.9706                                                                                              │
│  p-value    0.867                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 5. Post-hoc tests

Post-hoc comparisons are done manually here; the dispatcher runs them
automatically when `auto_posthoc=True`.

In [16]:
# Tukey HSD (after one-way ANOVA with equal variance)
ph = ps.posthoc_tukey(df_3group, value="value", group="group")
ph.show()
ph.pairwise

╭─────────────────────────────────────────────────── Tukey HSD ───────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Tukey HSD                                                                                           │
│  Statistic  nan                                                                                                 │
│  p-value    nan                                                                                                 │
│            Post-hoc                                                                                             │
│ ┏━━━┳━━━┳━━━━━━━━┳━━━━━━━━━━━┓                                                                                  │
│ ┃ A ┃ B ┃ diff   ┃ p         ┃                                                                                  │
│ ┡━━━╇━━━╇━━━━━━━━╇━━━━━━━━━━━┩                                                                                  │
│ │ A │ B │ -1.604 │ 0.008484  │                                                                                  │
│ │ A │ C │ -3.872 │ 5.982e-10 │                                                                                  │
│ │ B │ C │ -2.268 │ 0.0001362 │                                                                                  │
│ └───┴───┴────────┴───────────┘                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

,A,B,diff,p
0,A,B,-1.604074,8.484064e-03
1,A,C,-3.872487,5.981925e-10
2,B,C,-2.268413,1.362418e-04


In [17]:
# Games-Howell (after Welch's ANOVA, unequal variance)
ph = ps.posthoc_games_howell(df_3group, value="value", group="group")
ph.pairwise

,A,B,diff,p
0,A,B,-1.604074,2.693686e-04
1,A,C,-3.872487,3.666115e-07
2,B,C,-2.268413,2.217467e-03


In [18]:
# Dunn's test (after Kruskal-Wallis, non-parametric)
# Default correction: Holm-Bonferroni
ph = ps.posthoc_dunn(df_3group, value="value", group="group")
ph.pairwise

,A,B,p,p_adjust
0,A,B,4.386404e-03,holm
1,A,C,5.136683e-08,holm
2,B,C,9.992863e-03,holm


You can also correct any pairwise table explicitly. This is useful when a table comes from another tool, or when you intentionally ran an unadjusted post-hoc procedure.

In [19]:
unadjusted = ps.posthoc_dunn(df_3group, value="value", group="group", p_adjust=None).pairwise
ps.adjust_pairwise(unadjusted, method="holm")

,A,B,p,p_adjust,p_adjusted,reject,p_adjust_method,p_adjust_alpha
0,A,B,2.193202e-03,none,4.386404e-03,True,holm,0.05
1,A,C,1.712228e-08,none,5.136683e-08,True,holm,0.05
2,B,C,9.992863e-03,none,9.992863e-03,True,holm,0.05


In [20]:
# Unadjusted Dunn's
ps.posthoc_dunn(df_3group, value="value", group="group", p_adjust=None).pairwise

,A,B,p,p_adjust
0,A,B,2.193202e-03,none
1,A,C,1.712228e-08,none
2,B,C,9.992863e-03,none


---

## 6. Two-way (factorial) ANOVA

When you have two or more factors, pass a list of column names as `group`:

In [21]:
df_twoway = pd.DataFrame(
    {
        "genotype": np.repeat(["WT", "KO"], 30),
        "treatment": np.tile(np.repeat(["saline", "drug"], 15), 2),
        "value": rng.normal(10, 2, 60),
    }
)
# Inject an interaction effect
df_twoway.loc[(df_twoway["genotype"] == "KO") & (df_twoway["treatment"] == "drug"), "value"] += 5

result = ps.anova_twoway(df_twoway, value="value", group=["genotype", "treatment"])
result.show()

╭───────────────────────────────────── Two-way ANOVA (genotype * treatment) ──────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Two-way ANOVA (genotype * treatment)                                                                │
│  Statistic  27.08                                                                                               │
│  p-value    <0.0001                                                                                             │
│  np2        0.326                                                                                               │
│                                  Details                                                                        │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓                                      │
│ ┃ Source               ┃ SS    ┃ DF ┃ MS    ┃ F     ┃ p_unc     ┃ np2    ┃                                      │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩                                      │
│ │ genotype             │ 59.04 │ 1  │ 59.04 │ 19.11 │ 5.418e-05 │ 0.2544 │                                      │
│ │ treatment            │ 147.6 │ 1  │ 147.6 │ 47.78 │ 4.835e-09 │ 0.4604 │                                      │
│ │ genotype * treatment │ 83.69 │ 1  │ 83.69 │ 27.08 │ 2.871e-06 │ 0.326  │                                      │
│ │ Residual             │ 173.1 │ 56 │ 3.09  │ nan   │ nan       │ nan    │                                      │
│ └──────────────────────┴───────┴────┴───────┴───────┴───────────┴────────┘                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
# Full ANOVA table
result.details

,Source,SS,DF,MS,F,p_unc,np2
0,genotype,59.040953,1,59.040953,19.105519,5.418255e-05,0.254382
1,treatment,147.642731,1,147.642731,47.776854,4.835191e-09,0.460381
2,genotype * treatment,83.691788,1,83.691788,27.082473,2.870670e-06,0.325971
3,Residual,173.054362,56,3.090256,NaN,NaN,NaN


The dispatcher also routes multi-factor designs automatically:

In [23]:
result = ps.test(df_twoway, value="value", group=["genotype", "treatment"])
result.show()

╭───────────────────────────────────── Two-way ANOVA (genotype * treatment) ──────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Two-way ANOVA (genotype * treatment)                                                                │
│  Statistic  27.08                                                                                               │
│  p-value    <0.0001                                                                                             │
│  np2        0.326                                                                                               │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.1539  not rejected                                                                 │
│  equal variance  levene    0.2382  not rejected                                                                 │
│                                  Details                                                                        │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓                                      │
│ ┃ Source               ┃ SS    ┃ DF ┃ MS    ┃ F     ┃ p_unc     ┃ np2    ┃                                      │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩                                      │
│ │ genotype             │ 59.04 │ 1  │ 59.04 │ 19.11 │ 5.418e-05 │ 0.2544 │                                      │
│ │ treatment            │ 147.6 │ 1  │ 147.6 │ 47.78 │ 4.835e-09 │ 0.4604 │                                      │
│ │ genotype * treatment │ 83.69 │ 1  │ 83.69 │ 27.08 │ 2.871e-06 │ 0.326  │                                      │
│ │ Residual             │ 173.1 │ 56 │ 3.09  │ nan   │ nan       │ nan    │                                      │
│ └──────────────────────┴───────┴────┴───────┴───────┴───────────┴────────┘                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 7. Wide format data

PyStars accepts wide format where each group has its own column:

In [24]:
df_wide = pd.DataFrame(
    {
        "mouse": ["m1", "m2", "m3", "m4", "m5"],
        "control": rng.normal(10, 2, 5),
        "drug_A": rng.normal(12, 2, 5),
        "drug_B": rng.normal(9, 2, 5),
    }
)

result = ps.test(
    df_wide, format="wide", groups=["control", "drug_A", "drug_B"], subject_index="mouse"
)
result.show()

╭───────────────────────────────────────────────── One-way ANOVA ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       One-way ANOVA                                                                                       │
│  Statistic  6.12                                                                                                │
│  p-value    0.01472                                                                                             │
│  np2        0.5049                                                                                              │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro   0.4224  not rejected                                                                 │
│  equal variance  levene     0.462  not rejected                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
# Direct test with wide format
ps.kruskal(df_wide, format="wide", groups=["control", "drug_A", "drug_B"]).show()

╭────────────────────────────────────────────── Kruskal-Wallis test ──────────────────────────────────────────────╮
│  Field            Value                                                                                         │
│  Test             Kruskal-Wallis test                                                                           │
│  Statistic        8.72                                                                                          │
│  p-value          0.01278                                                                                       │
│  epsilon_squared  0.6229                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 8. Working with results

Every function returns a `TestResult` object with several useful methods.

In [26]:
result = ps.test(df_2group, value="value", group="group")

# Inspect fields
print(f"Test:       {result.test_name}")
print(f"Statistic:  {result.statistic:.3f}")
print(f"p-value:    {result.p_value:.4f}")
print(f"Effect size: {result.effect_size}")

Test:       Student's t-test
Statistic:  -4.148
p-value:    0.0001
Effect size: {'cohen_d': 1.0709184658694884, 'CI95%': [-2.51, -0.88]}


In [27]:
# Tidy DataFrame for export / programmatic use
result.to_dataframe()

,test,statistic,p_value,cohen_d,CI95%_0,CI95%_1,normality_method,normality_statistic,normality_p,equal_variance_method,equal_variance_statistic,equal_variance_p,posthoc
0,Student's t-test,-4.147649,0.000111,1.070918,-2.51,-0.88,shapiro,0.965142,0.416055,levene,0.048031,0.827294,<NA>


In [28]:
# Combining multiple results
r_a = ps.ttest(df_2group, value="value", group="group")
r_b = ps.anova(df_3group, value="value", group="group")
r_c = ps.kruskal(df_3group, value="value", group="group")

ps.to_dataframe([r_a, r_b, r_c], p_adjust="holm")

,test,statistic,p_value,p_adjusted,reject,p_adjust_method,p_adjust_alpha,cohen_d,CI95%_0,CI95%_1,posthoc,np2,epsilon_squared
0,Welch's t-test,-4.147649,1.111906e-04,1.111906e-04,True,holm,0.05,1.070918,-2.51,-0.88,<NA>,NaN,NaN
1,One-way ANOVA,27.660031,1.223724e-09,3.671173e-09,True,holm,0.05,NaN,NaN,NaN,<NA>,0.434496,NaN
2,Kruskal-Wallis test,31.875200,1.197811e-07,2.395622e-07,True,holm,0.05,NaN,NaN,NaN,<NA>,NaN,0.430746


---

## 9. The decision flowchart at a glance

The dispatcher implements this logic:

| Data shape | Assumptions | Selected test |
|---|---|---|
| 2 groups, independent | normal + equal var | Student's t-test |
| 2 groups, independent | normal + unequal var | Welch's t-test |
| 2 groups, independent | non-normal / small n | Mann-Whitney U |
| 2 groups, paired | normal differences | Paired t-test |
| 2 groups, paired | non-normal differences | Wilcoxon signed-rank |
| >2 groups, one factor | normal + equal var | One-way ANOVA + Tukey HSD |
| >2 groups, one factor | normal + unequal var | Welch's ANOVA + Games-Howell |
| >2 groups, one factor | non-normal | Kruskal-Wallis + Dunn's test |
| >=2 factors | — | Two-way ANOVA (interaction reported) |

Post-hoc tests run only when the omnibus test is significant
(p < `alpha`) and `auto_posthoc=True`. Groups with fewer than 3
observations are treated as non-normal.